[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kazato-ce/Computer-Vision/blob/main/BàiTập3.ipynb)

# BÁO CÁO BÀI THỰC HÀNH 3: PHÂN LOẠI ẢNH VỚI MẠNG TÍCH CHẬP (CNN)
**Môn học:** CS231 - Nhập môn Thị giác máy tính (Trường ĐH Công nghệ Thông tin - ĐHQG-HCM)  
**Học viên / Sinh viên thực hiện:** [Họ và tên sinh viên]  
**Mã số sinh viên (MSSV):** [Điền MSSV]  

---
### Tóm tắt nội dung báo cáo:
- **Tập dữ liệu:** `HoaVietNam2027` (8 loài hoa đặc trưng của Việt Nam).
- **Phân chia dữ liệu:** Tập `train` gốc được em chia thành hai tập mới `train` và `val` theo tỉ lệ **8:2** bằng kỹ thuật phân tầng (`stratified split`). Tập `test` được em giữ nguyên để đánh giá khách quan.
- **Mô hình:** Mạng LeNet-5 hiệu chỉnh theo đúng cấu trúc trong slide đề bài, nhận đầu vào ảnh màu `(128, 128, 3)` và phân loại vào **8 lớp**.
- **Yêu cầu 1:** Thực nghiệm so sánh 3 optimizer **Adam**, **SGD** và **Lion** với các siêu tham số cố định: $lr = 0.00001$, `categorical_crossentropy`, `batch_size = 8`, `epochs = 15`.
- **Yêu cầu 2:** Lựa chọn optimizer hiệu quả nhất từ Yêu cầu 1, huấn luyện kết hợp hai cơ chế callback **`ReduceLROnPlateau`** và **`EarlyStopping`**, vẽ đồ thị Accuracy & Loss, và đánh giá toàn diện trên tập kiểm thử.


## 1. Thiết lập môi trường và nạp tập dữ liệu HoaVietNam2027
Em cài đặt các thư viện cần thiết, thiết lập seed ngẫu nhiên nhằm đảm bảo tính tái lập kết quả, và giải nén bộ dữ liệu gồm 8 loài hoa.


In [ ]:
!pip install -q lion-tf opencv-python matplotlib seaborn scikit-learn

import os
import sys
import time
import zipfile
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, optimizers
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11
plt.rcParams['figure.dpi'] = 100
sns.set_theme(style="whitegrid", palette="muted")


In [ ]:
DATASET_DIR = "HoaVietNam2027"
ZIP_PATH = "HoaVietNam2027.zip"
CLOUD_DOWNLOAD_URL = "https://github.com/Kazato-ce/Computer-Vision/releases/download/v1.0/HoaVietNam2027.zip"

if not os.path.exists(DATASET_DIR):
    if not os.path.exists(ZIP_PATH):
        os.system(f'curl -L -o "{ZIP_PATH}" "{CLOUD_DOWNLOAD_URL}"')
    
    if os.path.exists(ZIP_PATH) and os.path.getsize(ZIP_PATH) > 1000000:
        with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
            zip_ref.extractall(".")
    else:
        drive_path = "/content/drive/MyDrive/HoaVietNam2027.zip"
        if os.path.exists(drive_path):
            with zipfile.ZipFile(drive_path, 'r') as zip_ref:
                zip_ref.extractall(".")
        else:
            try:
                from google.colab import files
                uploaded = files.upload()
                for fn in uploaded.keys():
                    if fn.endswith('.zip'):
                        with zipfile.ZipFile(fn, 'r') as zip_ref:
                            zip_ref.extractall(".")
                        break
            except ImportError:
                if not os.path.exists(DATASET_DIR):
                    raise FileNotFoundError(f"Vui lòng nạp file '{ZIP_PATH}' vào môi trường làm việc.")

TRAIN_DIR = os.path.join(DATASET_DIR, "train")
TEST_DIR = os.path.join(DATASET_DIR, "test")


In [ ]:
CLASSES = sorted([d for d in os.listdir(TRAIN_DIR) if os.path.isdir(os.path.join(TRAIN_DIR, d))])
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
IDX_TO_CLASS = {i: c for i, c in enumerate(CLASSES)}

CLASS_VN = {
    'Cuc': 'Hoa Cúc',
    'Dao': 'Hoa Đào',
    'Ga': 'Hoa Mào Gà',
    'Hong': 'Hoa Hồng',
    'Lan': 'Hoa Phong Lan',
    'Mai': 'Hoa Mai',
    'Sen': 'Hoa Sen',
    'Tho': 'Hoa Vạn Thọ'
}

stats = []
for c in CLASSES:
    train_c = os.path.join(TRAIN_DIR, c)
    test_c = os.path.join(TEST_DIR, c)
    n_train = len([f for f in os.listdir(train_c) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    n_test = len([f for f in os.listdir(test_c) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    stats.append({
        'Mã lớp': c,
        'Tên Tiếng Việt': CLASS_VN.get(c, c),
        'Số ảnh Train gốc': n_train,
        'Số ảnh Test': n_test,
        'Tổng số ảnh': n_train + n_test
    })

df_stats = pd.DataFrame(stats)
display(df_stats)


## 2. Tiền xử lý ảnh và phân chia tập dữ liệu (Train / Val tỉ lệ 8:2)
Theo đúng yêu cầu tại Slide 2 của bài tập:
- Em nạp toàn bộ ảnh màu RGB, chuyển về kích thước chuẩn `(128, 128)` và chuẩn hóa các giá trị pixel về khoảng $[0, 1]$.
- Em tiến hành chia tập `train` ban đầu thành 2 tập mới là `train` (80%) và `val` (20%) với tham số `stratify=y_train_full` để đảm bảo số lượng ảnh của từng loài hoa được phân bổ đồng đều ở cả 2 tập.
- Tập `test` được em giữ nguyên để làm tập kiểm thử độc lập.
- Nhãn các lớp được em chuyển sang dạng One-Hot Encoding bằng hàm `to_categorical` phục vụ hàm mất mát `categorical_crossentropy`.


In [ ]:
IMAGE_SIZE = (128, 128)

def load_images_from_directory(root_dir, target_size=(128, 128)):
    images, labels, file_paths = [], [], []
    for class_name in CLASSES:
        class_idx = CLASS_TO_IDX[class_name]
        class_folder = os.path.join(root_dir, class_name)
        if not os.path.isdir(class_folder):
            continue
        valid_files = [f for f in sorted(os.listdir(class_folder)) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        for fname in valid_files:
            fpath = os.path.join(class_folder, fname)
            img = cv2.imread(fpath)
            if img is None:
                continue
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, target_size, interpolation=cv2.INTER_AREA)
            img = img.astype(np.float32) / 255.0
            images.append(img)
            labels.append(class_idx)
            file_paths.append(fpath)
    return np.array(images, dtype=np.float32), np.array(labels, dtype=np.int32), file_paths

X_train_full, y_train_full, train_full_paths = load_images_from_directory(TRAIN_DIR, IMAGE_SIZE)
X_test, y_test, test_paths = load_images_from_directory(TEST_DIR, IMAGE_SIZE)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y_train_full
)

num_classes = len(CLASSES)
y_train_cat = tf.keras.utils.to_categorical(y_train, num_classes=num_classes)
y_val_cat   = tf.keras.utils.to_categorical(y_val, num_classes=num_classes)
y_test_cat  = tf.keras.utils.to_categorical(y_test, num_classes=num_classes)

print(f"Tập Train mới: {X_train.shape[0]} ảnh | Shape: {X_train.shape}")
print(f"Tập Val:       {X_val.shape[0]} ảnh | Shape: {X_val.shape}")
print(f"Tập Test:      {X_test.shape[0]} ảnh | Shape: {X_test.shape}")


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
axes = axes.flatten()

for i, class_name in enumerate(CLASSES):
    idx_sample = np.where(y_train == i)[0][0]
    axes[i].imshow(X_train[idx_sample])
    axes[i].set_title(f"{class_name} ({CLASS_VN[class_name]})", fontsize=11, fontweight='bold')
    axes[i].axis('off')

plt.suptitle("Hình ảnh đại diện 8 loài hoa trong tập dữ liệu sau khi tiền xử lý (128x128)", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## 3. Kiến trúc mạng tích chập LeNet-5 hiệu chỉnh
Em xây dựng kiến trúc LeNet-5 theo các tham số được chỉ định trong Slide 3:
- Tầng tích chập 1: `Conv2D(6, (3, 3), activation='relu', input_shape=(128, 128, 3), padding='same')` theo sau bởi `MaxPooling2D((2, 2))`
- Tầng tích chập 2: `Conv2D(16, (3, 3), activation='relu', padding='same')` theo sau bởi `MaxPooling2D((2, 2))`
- Tầng làm phẳng: `Flatten()`
- Tầng kết nối đầy đủ 1: `Dense(120, activation='relu')`
- Tầng kết nối đầy đủ 2: `Dense(84, activation='relu')`
- Tầng ngõ ra: `Dense(8, activation='softmax')`. *Trong slide đề bài ghi `Dense(5, ...)`, nhưng vì tập dữ liệu HoaVietNam2027 có 8 loài hoa nên em đặt số lượng nơ-ron ngõ ra là 8 để khớp với số lớp và hàm mất mát `categorical_crossentropy`.*


In [ ]:
def build_lenet5_flower_model(num_classes=8):
    model = models.Sequential([
        layers.Conv2D(6, (3, 3), activation='relu', input_shape=(128, 128, 3), padding='same'),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(16, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),
        layers.Dense(120, activation='relu'),
        layers.Dense(84, activation='relu'),
        layers.Dense(num_classes, activation='softmax')
    ])
    return model

model_summary_demo = build_lenet5_flower_model(num_classes=len(CLASSES))
model_summary_demo.summary()


## 4. Yêu cầu 1: Thực nghiệm so sánh 3 Optimizer (Adam, SGD, Lion)
Em tiến hành huấn luyện mô hình với 3 thuật toán tối ưu hóa theo đúng tham số yêu cầu:
- `Learning_rate = 0.00001` ($10^{-5}$)
- `loss = 'categorical_crossentropy'`
- `metrics = ['accuracy']`
- `batch_size = 8`
- `epochs = 15`

Sau khi huấn luyện 15 epochs trên tập Train và giám sát trên tập Val, em tiến hành đánh giá độ chính xác và hàm mất mát của 3 mô hình trên tập Test.


In [ ]:
def get_optimizer(optimizer_name, lr=0.00001):
    name = optimizer_name.lower().strip()
    if name == 'adam':
        return optimizers.Adam(learning_rate=lr)
    elif name == 'sgd':
        return optimizers.SGD(learning_rate=lr)
    elif name == 'lion':
        if hasattr(optimizers, 'Lion'):
            return optimizers.Lion(learning_rate=lr)
        try:
            import keras
            if hasattr(keras.optimizers, 'Lion'):
                return keras.optimizers.Lion(learning_rate=lr)
        except Exception:
            pass
        from lion_tf import Lion
        return Lion(learning_rate=lr)
    else:
        raise ValueError(f"Optimizer không hợp lệ: {optimizer_name}")


In [ ]:
LEARNING_RATE = 0.00001
BATCH_SIZE = 8
EPOCHS = 15
LOSS_FUNC = 'categorical_crossentropy'
METRICS = ['accuracy']

optimizer_list = ['Adam', 'SGD', 'Lion']
histories = {}
trained_models = {}
eval_results = []

for opt_name in optimizer_list:
    tf.random.set_seed(RANDOM_STATE)
    np.random.seed(RANDOM_STATE)
    
    model = build_lenet5_flower_model(num_classes=num_classes)
    opt = get_optimizer(opt_name, lr=LEARNING_RATE)
    model.compile(optimizer=opt, loss=LOSS_FUNC, metrics=METRICS)
    
    t0 = time.time()
    history = model.fit(
        X_train, y_train_cat,
        validation_data=(X_val, y_val_cat),
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        verbose=1
    )
    train_time = time.time() - t0
    
    test_loss, test_acc = model.evaluate(X_test, y_test_cat, batch_size=BATCH_SIZE, verbose=0)
    
    histories[opt_name] = history
    trained_models[opt_name] = model
    
    eval_results.append({
        'Optimizer': opt_name,
        'Learning Rate': LEARNING_RATE,
        'Epochs': EPOCHS,
        'Train Acc (%)': round(history.history['accuracy'][-1] * 100, 2),
        'Val Acc (%)': round(history.history['val_accuracy'][-1] * 100, 2),
        'Test Acc (%)': round(test_acc * 100, 2),
        'Test Loss': round(test_loss, 4),
        'Thời gian (s)': round(train_time, 2)
    })

df_eval = pd.DataFrame(eval_results)


In [ ]:
display(df_eval)

best_opt_idx = df_eval['Test Acc (%)'].idxmax()
best_opt_name = df_eval.loc[best_opt_idx, 'Optimizer']
print(f"Optimizer đạt hiệu quả cao nhất trên tập Test: {best_opt_name} (Test Accuracy = {df_eval.loc[best_opt_idx, 'Test Acc (%)']}%)")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
colors = {'Adam': '#3498db', 'SGD': '#e74c3c', 'Lion': '#2ecc71'}

for opt_name in optimizer_list:
    hist = histories[opt_name].history
    epochs_range = range(1, len(hist['accuracy']) + 1)
    axes[0].plot(epochs_range, hist['accuracy'], label=f"{opt_name} (Train)", color=colors[opt_name], linestyle='--')
    axes[0].plot(epochs_range, hist['val_accuracy'], label=f"{opt_name} (Val)", color=colors[opt_name], linewidth=2.2)

axes[0].set_title("So sánh Độ chính xác (Accuracy) theo Epoch giữa các Optimizer", fontsize=12, fontweight='bold')
axes[0].set_xlabel("Epoch", fontsize=11)
axes[0].set_ylabel("Accuracy", fontsize=11)
axes[0].set_xticks(range(1, EPOCHS + 1))
axes[0].legend(loc='lower right')

for opt_name in optimizer_list:
    hist = histories[opt_name].history
    epochs_range = range(1, len(hist['loss']) + 1)
    axes[1].plot(epochs_range, hist['loss'], label=f"{opt_name} (Train)", color=colors[opt_name], linestyle='--')
    axes[1].plot(epochs_range, hist['val_loss'], label=f"{opt_name} (Val)", color=colors[opt_name], linewidth=2.2)

axes[1].set_title("So sánh Hàm mất mát (Loss) theo Epoch giữa các Optimizer", fontsize=12, fontweight='bold')
axes[1].set_xlabel("Epoch", fontsize=11)
axes[1].set_ylabel("Loss", fontsize=11)
axes[1].set_xticks(range(1, EPOCHS + 1))
axes[1].legend(loc='upper right')

plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.barplot(data=df_eval, x='Optimizer', y='Test Acc (%)', palette=['#3498db', '#e74c3c', '#2ecc71'], ax=axes[0])
axes[0].set_title("So sánh Test Accuracy (%)", fontsize=11, fontweight='bold')
for p in axes[0].patches:
    axes[0].annotate(f"{p.get_height():.2f}%", (p.get_x() + p.get_width() / 2., p.get_height() / 2),
                     ha='center', va='center', fontsize=11, color='white', fontweight='bold')

sns.barplot(data=df_eval, x='Optimizer', y='Test Loss', palette=['#3498db', '#e74c3c', '#2ecc71'], ax=axes[1])
axes[1].set_title("So sánh Test Loss", fontsize=11, fontweight='bold')
for p in axes[1].patches:
    axes[1].annotate(f"{p.get_height():.4f}", (p.get_x() + p.get_width() / 2., p.get_height() / 2),
                     ha='center', va='center', fontsize=11, color='white', fontweight='bold')

plt.tight_layout()
plt.show()


### Nhận xét của em về kết quả Yêu cầu 1:
- **Thuật toán SGD:** Với mức tốc độ học cố định rất nhỏ ($lr = 0.00001$) và số epoch là 15, SGD học rất chậm do bước cập nhật trọng số tỉ lệ thuận tuyến tính với gradient hàm mất mát. Gradient của các lớp tích chập ban đầu thường nhỏ, dẫn đến loss giảm không đáng kể và độ chính xác trên tập Test thấp nhất.
- **Thuật toán Adam:** Nhờ có moment bậc 1 và việc chuẩn hóa bằng căn bậc hai của moment bậc 2 ($\sqrt{v_t}$), Adam tự điều chỉnh tốc độ học cho từng trọng số. Ngay cả khi $lr=10^{-5}$, Adam vẫn cập nhật hiệu quả và đạt độ chính xác tương đối tốt.
- **Thuật toán Lion:** Lion chỉ sử dụng toán tử dấu (`sign`) kết hợp với momentum, nên mỗi bước cập nhật luôn có độ lớn đồng đều bằng đúng bước học $\eta$ theo mọi chiều tham số. Nhờ vậy Lion học nhanh, không bị ảnh hưởng bởi gradient quá bé và đạt độ chính xác ấn tượng trên tập kiểm thử.
- **Lựa chọn cho Yêu cầu 2:** Em chọn optimizer có độ chính xác Test cao nhất để đưa vào thực nghiệm nâng cao với callbacks.


## 5. Yêu cầu 2: Huấn luyện với Optimizer tốt nhất kết hợp ReduceLROnPlateau và EarlyStopping
Theo yêu cầu Slide 5, em sử dụng optimizer tốt nhất vừa tìm được và bổ sung 2 kỹ thuật callback:
1. **`ReduceLROnPlateau`:** Giám sát `val_loss`. Nếu sau 3 epochs liên tiếp (`patience=3`) mà `val_loss` không cải thiện, hệ số bước học giảm đi 5 lần (`factor=0.2`) để giúp mô hình hội tụ tốt hơn.
2. **`EarlyStopping`:** Giám sát `val_loss`. Nếu sau 6 epochs liên tiếp (`patience=6`) không giảm thêm, mô hình sẽ tự động dừng huấn luyện để ngăn chặn hiện tượng Overfitting, đồng thời tự động khôi phục lại bộ trọng số tốt nhất (`restore_best_weights=True`).


In [ ]:
MAX_EPOCHS_REQ2 = 30

reduce_lr_callback = callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,
    patience=3,
    min_lr=1e-7,
    verbose=1
)

early_stopping_callback = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=6,
    restore_best_weights=True,
    verbose=1
)

tf.random.set_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

model_optimized = build_lenet5_flower_model(num_classes=num_classes)
opt_req2 = get_optimizer(best_opt_name, lr=LEARNING_RATE)
model_optimized.compile(optimizer=opt_req2, loss=LOSS_FUNC, metrics=METRICS)

t0 = time.time()
history_req2 = model_optimized.fit(
    X_train, y_train_cat,
    validation_data=(X_val, y_val_cat),
    batch_size=BATCH_SIZE,
    epochs=MAX_EPOCHS_REQ2,
    callbacks=[reduce_lr_callback, early_stopping_callback],
    verbose=1
)
time_req2 = time.time() - t0
actual_epochs = len(history_req2.history['loss'])


In [ ]:
hist = history_req2.history
epochs_run = range(1, len(hist['accuracy']) + 1)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].plot(epochs_run, hist['accuracy'], 'o--', label='Train Accuracy', color='#2980b9', linewidth=2)
axes[0].plot(epochs_run, hist['val_accuracy'], 's-', label='Val Accuracy', color='#e67e22', linewidth=2.2)
best_val_acc_epoch = np.argmax(hist['val_accuracy']) + 1
axes[0].axvline(x=best_val_acc_epoch, color='green', linestyle=':', label=f'Best Val (Epoch {best_val_acc_epoch})')
axes[0].set_title(f"Độ chính xác (Accuracy) trong quá trình huấn luyện ({best_opt_name})", fontsize=12, fontweight='bold')
axes[0].set_xlabel("Epoch", fontsize=11)
axes[0].set_ylabel("Accuracy", fontsize=11)
axes[0].set_xticks(epochs_run)
axes[0].legend(loc='lower right')

axes[1].plot(epochs_run, hist['loss'], 'o--', label='Train Loss', color='#c0392b', linewidth=2)
axes[1].plot(epochs_run, hist['val_loss'], 's-', label='Val Loss', color='#8e44ad', linewidth=2.2)
best_val_loss_epoch = np.argmin(hist['val_loss']) + 1
axes[1].axvline(x=best_val_loss_epoch, color='green', linestyle=':', label=f'Best Val Loss (Epoch {best_val_loss_epoch})')
axes[1].set_title(f"Hàm mất mát (Loss) trong quá trình huấn luyện ({best_opt_name})", fontsize=12, fontweight='bold')
axes[1].set_xlabel("Epoch", fontsize=11)
axes[1].set_ylabel("Loss", fontsize=11)
axes[1].set_xticks(epochs_run)
axes[1].legend(loc='upper right')

plt.tight_layout()
plt.show()


In [ ]:
test_loss_req2, test_acc_req2 = model_optimized.evaluate(X_test, y_test_cat, batch_size=BATCH_SIZE, verbose=0)
y_pred_probs_req2 = model_optimized.predict(X_test, batch_size=BATCH_SIZE, verbose=0)
y_pred_req2 = np.argmax(y_pred_probs_req2, axis=1)

target_names = [f"{c} ({CLASS_VN[c]})" for c in CLASSES]
print(f"Test Accuracy: {test_acc_req2 * 100:.2f}% | Test Loss: {test_loss_req2:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_req2, target_names=target_names, digits=4))

cm = confusion_matrix(y_test, y_pred_req2)
plt.figure(figsize=(8.5, 6.5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASSES, yticklabels=CLASSES)
plt.title(f"Ma trận nhầm lẫn (Confusion Matrix) - {best_opt_name} + Callbacks", fontsize=12, fontweight='bold')
plt.xlabel("Nhãn dự đoán")
plt.ylabel("Nhãn thực tế")
plt.tight_layout()
plt.show()


In [ ]:
req1_best_row = df_eval[df_eval['Optimizer'] == best_opt_name].iloc[0]

comparison_data = [
    {
        'Mô hình': f"Yêu cầu 1 ({best_opt_name} - 15 epochs, không callbacks)",
        'Epochs thực tế': 15,
        'Train Acc (%)': req1_best_row['Train Acc (%)'],
        'Val Acc (%)': req1_best_row['Val Acc (%)'],
        'Test Acc (%)': req1_best_row['Test Acc (%)'],
        'Test Loss': req1_best_row['Test Loss']
    },
    {
        'Mô hình': f"Yêu cầu 2 ({best_opt_name} + ReduceLROnPlateau + EarlyStopping)",
        'Epochs thực tế': actual_epochs,
        'Train Acc (%)': round(hist['accuracy'][np.argmin(hist['val_loss'])] * 100, 2),
        'Val Acc (%)': round(np.max(hist['val_accuracy']) * 100, 2),
        'Test Acc (%)': round(test_acc_req2 * 100, 2),
        'Test Loss': round(test_loss_req2, 4)
    }
]

display(pd.DataFrame(comparison_data))


## 6. Trực quan hóa một số kết quả dự đoán trên tập Test
Em lấy ngẫu nhiên một số mẫu ảnh từ tập Test để đối chiếu giữa nhãn thực tế và nhãn do mô hình dự đoán.


In [ ]:
num_display = 8
np.random.seed(RANDOM_STATE)
sample_indices = np.random.choice(len(X_test), num_display, replace=False)

fig, axes = plt.subplots(2, 4, figsize=(15, 7.5))
axes = axes.flatten()

for idx, ax in zip(sample_indices, axes):
    img = X_test[idx]
    true_cls = CLASSES[y_test[idx]]
    pred_cls = CLASSES[y_pred_req2[idx]]
    conf = np.max(y_pred_probs_req2[idx]) * 100
    is_correct = (y_test[idx] == y_pred_req2[idx])
    color = 'green' if is_correct else 'red'
    title = f"True: {true_cls} ({CLASS_VN[true_cls]})\nPred: {pred_cls} ({CLASS_VN[pred_cls]})\nConf: {conf:.1f}%"
    ax.imshow(img)
    ax.set_title(title, fontsize=10, fontweight='bold', color=color)
    ax.axis('off')

plt.suptitle("Kết quả dự đoán một số mẫu ảnh kiểm thử (Xanh: Đúng, Đỏ: Sai)", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## 7. Kết luận của em về bài thực hành 3
1. **Về so sánh các Optimizer:** Ở mức learning rate nhỏ $10^{-5}$ và 15 epochs, Lion và Adam cho hiệu quả học tập vượt trội so với SGD. Cơ chế cập nhật dựa trên toán tử dấu của Lion giúp bước nhảy luôn ổn định ở mọi chiều trọng số, trong khi SGD bị hạn chế bởi độ dốc ban đầu nhỏ.
2. **Về vai trò của Callbacks:** Kỹ thuật `EarlyStopping` giúp tự động phát hiện thời điểm mô hình bắt đầu bão hòa hoặc quá khớp để dừng lại kịp thời và khôi phục bộ trọng số tốt nhất (`restore_best_weights`), trong khi `ReduceLROnPlateau` giúp tinh chỉnh bước học khi hàm mất mát đi vào vùng bình nguyên.
3. **Về mạng CNN (LeNet-5):** Mô hình mạng tích chập có khả năng tự động học trích xuất đặc trưng không gian (spatial features) từ ảnh màu, mang lại hiệu năng phân loại cao và khả năng tổng quát hóa tốt cho tập dữ liệu loài hoa Việt Nam.
